In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix)

In [2]:
RANDOM_STATE = 42
TARGET = "SeriousDlqin2yrs"
path = sys.argv[1] if len(sys.argv) > 1 else "cs-training.csv"

In [4]:
# ---------------------------------------------------------------
# STEP 1: Load data
# ---------------------------------------------------------------
df = pd.read_csv("cs-training.csv")
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")])
print("Original shape:", df.shape)
print("Default rate: %.1f%%" % (df[TARGET].mean() * 100))
print("\nMissing values:\n", df.isna().sum()[df.isna().sum() > 0])

Original shape: (150000, 11)
Default rate: 6.7%

Missing values:
 MonthlyIncome         29731
NumberOfDependents     3924
dtype: int64


In [5]:
# ---------------------------------------------------------------
# STEP 2: Cleaning (this dataset has some known problems)
# ---------------------------------------------------------------
# (a) Impossible ages
df = df[df["age"] >= 18].copy()

# (b) Late-payment columns use 96 and 98 as error codes -> treat as missing
late_cols = ["NumberOfTime30-59DaysPastDueNotWorse",
             "NumberOfTimes90DaysLate",
             "NumberOfTime60-89DaysPastDueNotWorse"]
for c in late_cols:
    df.loc[df[c] >= 90, c] = np.nan

# (c) Extreme outliers: cap at the 99th percentile
for c in ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio", "MonthlyIncome"]:
    df[c] = df[c].clip(upper=df[c].quantile(0.99))

In [6]:
# ---------------------------------------------------------------
# STEP 3: Feature engineering
# ---------------------------------------------------------------
df["income_missing"] = df["MonthlyIncome"].isna().astype(int)
df["total_late"] = df[late_cols].sum(axis=1)                  # all late payments together
df["ever_90_late"] = (df["NumberOfTimes90DaysLate"] > 0).astype(int)
df["log_income"] = np.log1p(df["MonthlyIncome"])
df["monthly_debt"] = df["DebtRatio"] * df["MonthlyIncome"]    # DebtRatio is a ratio, so convert to an amount
df["income_per_person"] = df["MonthlyIncome"] / (df["NumberOfDependents"].fillna(0) + 1)
df["high_utilization"] = (df["RevolvingUtilizationOfUnsecuredLines"] > 0.8).astype(int)
df["age_group"] = pd.cut(df["age"], [17, 30, 45, 60, 120], labels=False)


In [7]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)


In [8]:
# ---------------------------------------------------------------
# STEP 4: Models (imputation happens inside the pipeline,
#         so test data never leaks into training)
# ---------------------------------------------------------------
models = {
    "Logistic Regression": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ]),
    "Decision Tree": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(max_depth=6, min_samples_leaf=50,
                                         class_weight="balanced",
                                         random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=200, min_samples_leaf=20,
                                         class_weight="balanced_subsample",
                                         random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
}

In [12]:
# ---------------------------------------------------------------
# STEP 5: Train and evaluate
# ---------------------------------------------------------------
rows = []
plt.figure(figsize=(6, 5))
for name, pipe in models.items():
    print("\nTraining", name, "...")
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]

    rows.append({
        "Model": name,
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
    })
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))

    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, label="%s (AUC=%.3f)" % (name, rows[-1]["ROC-AUC"]))

plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - Give Me Some Credit")
plt.legend()
plt.tight_layout()
plt.savefig("roc_curves.png", dpi=150)

results = pd.DataFrame(rows).set_index("Model").round(3)
print("\n=== Model comparison ===")
print(results)
results.to_csv("model_results.csv")


Training Logistic Regression ...
Confusion matrix:
 [[22530  5465]
 [  481  1524]]

Training Decision Tree ...
Confusion matrix:
 [[21021  6974]
 [  383  1622]]

Training Random Forest ...
Confusion matrix:
 [[24725  3270]
 [  666  1339]]

=== Model comparison ===
                     Precision  Recall     F1  ROC-AUC
Model                                                 
Logistic Regression      0.218   0.760  0.339    0.861
Decision Tree            0.189   0.809  0.306    0.854
Random Forest            0.291   0.668  0.405    0.864


In [10]:
# ---------------------------------------------------------------
# STEP 6: Feature importance (Random Forest)
# ---------------------------------------------------------------

rf = models["Random Forest"].named_steps["model"]
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(7, 6))
importance.plot(kind="barh")
plt.title("Feature importance (Random Forest)")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
print("\nTop 5 features:\n", importance.sort_values(ascending=False).head(5).round(3))


Top 5 features:
 RevolvingUtilizationOfUnsecuredLines    0.216
total_late                              0.165
high_utilization                        0.077
NumberOfTime30-59DaysPastDueNotWorse    0.059
DebtRatio                               0.057
dtype: float64


In [13]:
print(y.value_counts())
print(y.value_counts(normalize=True).round(4))

SeriousDlqin2yrs
0    139973
1     10026
Name: count, dtype: int64
SeriousDlqin2yrs
0    0.9332
1    0.0668
Name: proportion, dtype: float64


In [18]:

from sklearn.base import clone
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import numpy as np
import pandas as pd

candidates = []

# Model and threshold selection: validation data only
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    val_prob = pipe.predict_proba(X)[:, 1]

    best = None
    for threshold in np.arange(0.10, 0.71, 0.02):
        val_pred = (val_prob >= threshold).astype(int)
        f1 = f1_score(y, val_pred, zero_division=0)

        if best is None or f1 > best["F1"]:
            best = {
                "Model": name,
                "Threshold": float(threshold),
                "F1": f1,
                "Precision": precision_score(
                    y, val_pred, zero_division=0
                ),
                "Recall": recall_score(
                    y, val_pred, zero_division=0
                ),
                "ROC-AUC": roc_auc_score(y, val_prob),
            }

    candidates.append(best)

validation_results = pd.DataFrame(candidates)
print("Validation results:")
print(validation_results.round(3).to_string(index=False))

# Choose based on validation F1
winner = validation_results.loc[validation_results["F1"].idxmax()]
best_name = winner["Model"]
best_threshold = winner["Threshold"]

# Retrain the selected model on train + validation data
final_model = clone(models[best_name])
final_model.fit(X_train, y_train)

# Final evaluation: untouched test set
test_prob = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= best_threshold).astype(int)

print("\nSelected model:", best_name)
print("Selected threshold:", round(best_threshold, 2))
print("\nTest precision:",
      round(precision_score(y_test, test_pred, zero_division=0), 3))
print("Test recall:",
      round(recall_score(y_test, test_pred, zero_division=0), 3))
print("Test F1:",
      round(f1_score(y_test, test_pred, zero_division=0), 3))
print("Test ROC-AUC:", round(roc_auc_score(y_test, test_prob), 3))
print("\nConfusion matrix:\n", confusion_matrix(y_test, test_pred))
print("\nClassification report:\n",
      classification_report(y_test, test_pred, zero_division=0))


Validation results:
              Model  Threshold    F1  Precision  Recall  ROC-AUC
Logistic Regression       0.70 0.426      0.357   0.527    0.856
      Decision Tree       0.70 0.425      0.337   0.576    0.859
      Random Forest       0.66 0.499      0.417   0.621    0.929

Selected model: Random Forest
Selected threshold: 0.66

Test precision: 0.372
Test recall: 0.539
Test F1: 0.44
Test ROC-AUC: 0.864

Confusion matrix:
 [[26175  1820]
 [  925  1080]]

Classification report:
               precision    recall  f1-score   support

           0       0.97      0.93      0.95     27995
           1       0.37      0.54      0.44      2005

    accuracy                           0.91     30000
   macro avg       0.67      0.74      0.70     30000
weighted avg       0.93      0.91      0.92     30000



In [20]:

import joblib

artifact = {
    "model": final_model,
    "features": list(X_train.columns),
    "threshold": float(best_threshold),
}

joblib.dump(artifact, "artifacts/credit_model.joblib")

print("Model saved successfully!")


Model saved successfully!
